###03 - Camada Gold

A camada Gold é a **vitrine** do projeto.

Agora combinamos as informações para responder perguntas de negócio. Em vez de trabalhar com 24 tabelas separadas, criamos bases analíticas simples e prontas para consumo.

In [0]:
#Configuração: definimos origem dos dados e o destino das tabelas analíticas

from pyspark.sql import functions as F

CATALOG = "nautical_lighthouse"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"


In [0]:
#Base analítica de vendas

orders = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.orders"
)

order_items = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.order_items"
)

customers = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.customers"
)

variants = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.product_variants"
)

products = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.products"
)

brands = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.brands"
)

categories = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.categories"
)

In [0]:
#Conferência de estrutura

tables_to_check = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "product_variants": variants,
    "products": products,
    "brands": brands,
    "categories": categories
}

for name, df in tables_to_check.items():
    print(f"\n--- {name} ---")
    print(df.columns)

In [0]:
#Separar venda válida de pedido cancelado

orders_prepared = (
    orders
    .withColumn(
        "is_cancelled",
        F.col("status") == "cancelled"
    )
    .withColumn(
        "order_date",
        F.to_date("placed_at")
    )
    .withColumn(
        "order_year",
        F.year("placed_at")
    )
    .withColumn(
        "order_month",
        F.month("placed_at")
    )
)

display(
    orders_prepared
    .groupBy("status", "is_cancelled")
    .count()
    .orderBy("status")
)

In [0]:
#Calendário completo, com todos os dias do período analisado

date_range = orders.select(
    F.min(F.to_date("placed_at")).alias("min_date"),
    F.max(F.to_date("placed_at")).alias("max_date")
).first()

calendar = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{date_range["min_date"]}'),
                to_date('{date_range["max_date"]}'),
                interval 1 day
            )
        ) AS date
        """
    )
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn(
        "day_of_week_number",
        F.dayofweek("date")
    )
    .withColumn(
        "day_of_week",
        F.date_format("date", "EEEE")
    )
)

display(calendar.limit(10))

In [0]:
#Calendário analítico

(
    calendar.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.dim_calendar"
    )
)

print(
    f"Dias no calendário: {calendar.count():,}"
)

In [0]:
#Validação do calendário

calendar_validation = calendar.select(
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date"),
    F.count("*").alias("total_days"),
    F.countDistinct("date").alias("unique_days")
)

display(calendar_validation)

In [0]:
#Produtos prontos para análise

dim_product = (
    variants.alias("v")
    .join(
        products.alias("p"),
        F.col("v.product_id") == F.col("p.id"),
        "left"
    )
    .join(
        brands.alias("b"),
        F.col("p.brand_id") == F.col("b.id"),
        "left"
    )
    .join(
        categories.alias("c"),
        F.col("p.category_id") == F.col("c.id"),
        "left"
    )
    .select(
        F.col("v.id").alias("product_variant_id"),
        F.col("v.product_id"),
        F.col("v.sku"),
        F.col("p.name").alias("product_name"),
        F.col("b.name").alias("brand_name"),
        F.col("c.name").alias("category_name"),
        F.col("v.sale_price"),
        F.col("v.cost_price"),
        F.col("v.is_active").alias("variant_is_active")
    )
)

display(dim_product.limit(10))

In [0]:
#Salvando a dimensão de produtos

(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.dim_product"
    )
)

print(f"Produtos/variantes na Gold: {dim_product.count():,}")

In [0]:
#Base principal de vendas

fact_sales = (
    order_items.alias("oi")
    .join(
        orders_prepared.alias("o"),
        F.col("oi.order_id") == F.col("o.id"),
        "left"
    )
    .join(
        customers.alias("cu"),
        F.col("o.customer_id") == F.col("cu.id"),
        "left"
    )
    .join(
        dim_product.alias("dp"),
        F.col("oi.product_variant_id") == F.col("dp.product_variant_id"),
        "left"
    )
    .select(
        F.col("oi.id").alias("order_item_id"),
        F.col("oi.order_id"),
        F.col("o.order_number"),
        F.col("o.order_date"),
        F.col("o.order_year"),
        F.col("o.order_month"),
        F.col("o.channel"),
        F.col("o.status"),
        F.col("o.is_cancelled"),
        F.col("o.customer_id"),
        F.col("cu.legal_name").alias("customer_name"),
        F.col("oi.product_variant_id"),
        F.col("dp.sku"),
        F.col("dp.product_name"),
        F.col("dp.brand_name"),
        F.col("dp.category_name"),
        F.col("oi.quantity"),
        F.col("oi.unit_price"),
        F.col("dp.cost_price"),
        F.col("oi.line_total")
    )
)

In [0]:
#Receita, custo e lucro

fact_sales = (
    fact_sales
    .withColumn(
        "gross_revenue",
        F.col("line_total")
    )
    .withColumn(
        "total_cost",
        F.round(
            F.col("quantity") * F.col("cost_price"),
            2
        )
    )
    .withColumn(
        "gross_profit",
        F.round(
            F.col("gross_revenue") - F.col("total_cost"),
            2
        )
    )
    .withColumn(
        "gross_margin_pct",
        F.when(
            F.col("gross_revenue") != 0,
            F.round(
                F.col("gross_profit") / F.col("gross_revenue") * 100,
                2
            )
        )
    )
)

display(fact_sales.limit(10))

In [0]:
#Vendas válidas, sem misturar com intenção de compra

fact_sales_valid = (
    fact_sales
    .filter(~F.col("is_cancelled"))
)

print(f"Itens totais: {fact_sales.count():,}")
print(f"Itens de vendas válidas: {fact_sales_valid.count():,}")


In [0]:
#Salvando a base principal

(
    fact_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.fact_sales"
    )
)

In [0]:
#Conferência da Gold

gold_reconciliation = (
    fact_sales
    .groupBy("order_id")
    .agg(
        F.round(
            F.sum("gross_revenue"),
            2
        ).alias("gold_items_total")
    )
    .join(
        orders.select(
            F.col("id").alias("order_id"),
            "subtotal"
        ),
        on="order_id",
        how="left"
    )
    .withColumn(
        "difference",
        F.round(
            F.col("gold_items_total") - F.col("subtotal"),
            2
        )
    )
)

invalid_gold_orders = (
    gold_reconciliation
    .filter(F.abs(F.col("difference")) > 0.01)
    .count()
)

print(
    f"Pedidos com divergência na Gold: {invalid_gold_orders}"
)

In [0]:
#Valor gerado por cliente

customer_summary = (
    fact_sales_valid
    .groupBy(
        "customer_id",
        "customer_name"
    )
    .agg(
        F.countDistinct("order_id").alias("orders_count"),
        F.round(F.sum("gross_revenue"), 2).alias("revenue"),
        F.round(F.sum("total_cost"), 2).alias("cost"),
        F.round(F.sum("gross_profit"), 2).alias("profit")
    )
    .withColumn(
        "margin_pct",
        F.when(
            F.col("revenue") != 0,
            F.round(
                F.col("profit") / F.col("revenue") * 100,
                2
            )
        )
    )
)

display(
    customer_summary
    .orderBy(F.desc("profit"))
    .limit(20)
)

In [0]:
#Salvando a visão de clientes

(
    customer_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.customer_profitability"
    )
)

In [0]:
#Vendas por dia, inclusive quando não houve venda

daily_sales = (
    fact_sales_valid
    .groupBy("order_date")
    .agg(
        F.round(F.sum("gross_revenue"), 2).alias("daily_revenue"),
        F.countDistinct("order_id").alias("orders_count")
    )
)

daily_sales_complete = (
    calendar.alias("c")
    .join(
        daily_sales.alias("s"),
        F.col("c.date") == F.col("s.order_date"),
        "left"
    )
    .select(
        F.col("c.date"),
        F.col("c.year"),
        F.col("c.month"),
        F.col("c.day_of_week"),
        F.coalesce(
            F.col("s.daily_revenue"),
            F.lit(0.0)
        ).alias("daily_revenue"),
        F.coalesce(
            F.col("s.orders_count"),
            F.lit(0)
        ).alias("orders_count")
    )
)

display(daily_sales_complete.limit(10))

In [0]:
#Salvando as vendas diárias
(
    daily_sales_complete.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.daily_sales"
    )
)

In [0]:
#Dados de devolução

returns = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.returns"
)

return_items = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.return_items"
)

print(returns.columns)
print(return_items.columns)

In [0]:
#Base de devoluções

fact_returns = (
    return_items.alias("ri")
    .join(
        returns.alias("r"),
        F.col("ri.return_id") == F.col("r.id"),
        "left"
    )
    .join(
        order_items.alias("oi"),
        F.col("ri.order_item_id") == F.col("oi.id"),
        "left"
    )
    .join(
        dim_product.alias("dp"),
        F.col("oi.product_variant_id") == F.col("dp.product_variant_id"),
        "left"
    )
    .select(
        F.col("ri.id").alias("return_item_id"),
        F.col("ri.return_id"),
        F.col("r.return_number"),
        F.col("r.order_id"),
        F.col("r.customer_id"),
        F.to_date("r.created_at").alias("return_date"),
        F.col("r.status").alias("return_status"),
        F.col("r.reason"),
        F.col("ri.action"),
        F.col("oi.product_variant_id"),
        F.col("dp.sku"),
        F.col("dp.product_name"),
        F.col("dp.brand_name"),
        F.col("dp.category_name"),
        F.col("ri.quantity"),
        F.col("dp.cost_price"),
        F.col("ri.unit_refund_amount"),
        F.round(
            F.col("ri.quantity") * F.col("ri.unit_refund_amount"),
            2
        ).alias("refund_amount")
    )
)

display(fact_returns.limit(10))

In [0]:
#Conferência das devoluções

(
    fact_returns.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.fact_returns"
    )
)

returns_reconciliation = (
    fact_returns
    .groupBy("return_id")
    .agg(
        F.round(
            F.sum("refund_amount"),
            2
        ).alias("calculated_refund")
    )
    .join(
        returns.select(
            F.col("id").alias("return_id"),
            "total_refund_amount"
        ),
        on="return_id",
        how="left"
    )
    .withColumn(
        "difference",
        F.round(
            F.col("calculated_refund")
            - F.col("total_refund_amount"),
            2
        )
    )
)

invalid_returns = (
    returns_reconciliation
    .filter(F.abs(F.col("difference")) > 0.01)
    .count()
)

print(
    f"Devoluções com divergência financeira: "
    f"{invalid_returns:,}"
)

In [0]:
#Dados de estoque

stock_levels = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.stock_levels"
)

locations = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.locations"
)

print("--- stock_levels ---")
print(stock_levels.columns)

print("\n--- locations ---")
print(locations.columns)

In [0]:
#Base de estoque (qtd disponível x produto x local x custo)

fact_inventory = (
    stock_levels.alias("sl")
    .join(
        dim_product.alias("dp"),
        F.col("sl.product_variant_id") == F.col("dp.product_variant_id"),
        "left"
    )
    .join(
        locations.alias("l"),
        F.col("sl.location_id") == F.col("l.id"),
        "left"
    )
    .select(
        F.col("sl.product_variant_id"),
        F.col("dp.sku"),
        F.col("dp.product_name"),
        F.col("dp.brand_name"),
        F.col("dp.category_name"),
        F.col("sl.location_id"),
        F.col("l.name").alias("location_name"),
        F.col("sl.quantity_on_hand"),
        F.col("sl.reorder_point"),
        F.col("dp.cost_price"),
        F.col("dp.sale_price"),
        F.round(
            F.col("sl.quantity_on_hand") * F.col("dp.cost_price"),
            2
        ).alias("inventory_cost_value"),
        F.round(
            F.col("sl.quantity_on_hand") * F.col("dp.sale_price"),
            2
        ).alias("inventory_sale_value")
    )
)

display(fact_inventory.limit(10))

In [0]:
#Conferência de estoque

(
    fact_inventory.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.fact_inventory"
    )
)

source_inventory_rows = stock_levels.count()
gold_inventory_rows = fact_inventory.count()

print(f"Linhas na Silver: {source_inventory_rows:,}")
print(f"Linhas na Gold: {gold_inventory_rows:,}")
print(
    f"Validação: "
    f"{'OK' if source_inventory_rows == gold_inventory_rows else 'REVISAR'}"
)

In [0]:
#Valor financeiro do estoque

display(
    fact_inventory.agg(
        F.round(
            F.sum("inventory_cost_value"),
            2
        ).alias("total_inventory_cost"),
        F.round(
            F.sum("inventory_sale_value"),
            2
        ).alias("potential_sale_value")
    )
)

In [0]:
#Dados de compras

purchase_orders = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.purchase_orders"
)

purchase_order_items = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.purchase_order_items"
)

suppliers = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.suppliers"
)

product_suppliers = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.product_suppliers"
)

print("--- purchase_orders ---")
print(purchase_orders.columns)

print("\n--- purchase_order_items ---")
print(purchase_order_items.columns)

print("\n--- suppliers ---")
print(suppliers.columns)

print("\n--- product_suppliers ---")
print(product_suppliers.columns)

In [0]:
#Base de compras
fact_purchases = (
    purchase_order_items.alias("poi")
    .join(
        purchase_orders.alias("po"),
        F.col("poi.purchase_order_id") == F.col("po.id"),
        "left"
    )
    .join(
        suppliers.alias("s"),
        F.col("po.supplier_id") == F.col("s.id"),
        "left"
    )
    .join(
        dim_product.alias("dp"),
        F.col("poi.product_variant_id") == F.col("dp.product_variant_id"),
        "left"
    )
    .select(
        F.col("poi.id").alias("purchase_order_item_id"),
        F.col("po.id").alias("purchase_order_id"),
        F.col("po.po_number"),
        F.to_date("po.placed_at").alias("purchase_date"),
        F.col("po.status").alias("purchase_status"),
        F.col("po.currency"),
        F.col("po.supplier_id"),
        F.coalesce(
            F.col("s.trade_name"),
            F.col("s.legal_name")
        ).alias("supplier_name"),
        F.col("poi.product_variant_id"),
        F.col("dp.sku"),
        F.col("dp.product_name"),
        F.col("dp.brand_name"),
        F.col("dp.category_name"),
        F.col("poi.quantity_ordered"),
        F.col("poi.unit_cost"),
        F.col("poi.line_total")
    )
)

display(fact_purchases.limit(10))

In [0]:
#Conferência dos valores de compra

purchase_financial_check = (
    fact_purchases
    .withColumn(
        "expected_line_total",
        F.round(
            F.col("quantity_ordered") * F.col("unit_cost"),
            2
        )
    )
    .withColumn(
        "difference",
        F.round(
            F.col("line_total") - F.col("expected_line_total"),
            2
        )
    )
)

invalid_purchase_items = (
    purchase_financial_check
    .filter(F.abs(F.col("difference")) > 0.01)
    .count()
)

print(
    f"Itens de compra com divergência financeira: "
    f"{invalid_purchase_items:,}"
)

In [0]:
#Salvando a base de compras

(
    fact_purchases.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.fact_purchases"
    )
)

print(f"Itens de compra na Gold: {fact_purchases.count():,}")

In [0]:
#Condições dos fornecedores

supplier_conditions = (
    product_suppliers.alias("ps")
    .join(
        suppliers.alias("s"),
        F.col("ps.supplier_id") == F.col("s.id"),
        "left"
    )
    .join(
        dim_product.alias("dp"),
        F.col("ps.product_variant_id") == F.col("dp.product_variant_id"),
        "left"
    )
    .select(
        F.col("ps.product_variant_id"),
        F.col("dp.sku"),
        F.col("dp.product_name"),
        F.col("ps.supplier_id"),
        F.coalesce(
            F.col("s.trade_name"),
            F.col("s.legal_name")
        ).alias("supplier_name"),
        F.col("ps.last_quoted_cost"),
        F.col("ps.lead_time_days"),
        F.col("ps.is_preferred")
    )
)

display(supplier_conditions.limit(10))

In [0]:
#Salvar as condições dos fornecedores

(
    supplier_conditions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{GOLD_SCHEMA}.supplier_conditions"
    )
)

print(f"Condições dos fornecedores na Gold: {supplier_conditions.count():,}")

In [0]:
#Conferência final da Gold

gold_tables = spark.sql(
    f"SHOW TABLES IN {CATALOG}.{GOLD_SCHEMA}"
)

display(
    gold_tables.orderBy("tableName")
)

## Resultado da camada Gold

A Gold reúne as principais bases prontas para análise:

- vendas;
- clientes;
- produtos;
- devoluções;
- estoque;
- compras;
- fornecedores;
- calendário.

Essas tabelas serão usadas nas análises de negócio, previsão de demanda e sistema de recomendação.

A partir daqui, não precisamos voltar aos 24 arquivos originais para responder às principais perguntas do desafio.

In [0]:
%sql
SHOW TABLES IN nautical_lighthouse.gold;